# Notebook 3 - Model Training & Optimization
**Owner:** Member 3

**Models:**
1. Linear Regression (baseline) - trained on scaled features + log(charges)
2. Decision Tree Regressor - trained on unscaled features
3. Random Forest Regressor - trained on unscaled features


## 1. Import Libraries & Load Processed Data

In [1]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_DIR = "../data/processed"
MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)

X_train = pd.read_csv(f"{DATA_DIR}/X_train.csv")
X_test = pd.read_csv(f"{DATA_DIR}/X_test.csv")
X_train_scaled = pd.read_csv(f"{DATA_DIR}/X_train_scaled.csv")
X_test_scaled = pd.read_csv(f"{DATA_DIR}/X_test_scaled.csv")

y_train = pd.read_csv(f"{DATA_DIR}/y_train.csv").squeeze()
y_test = pd.read_csv(f"{DATA_DIR}/y_test.csv").squeeze()

print("Train:", X_train.shape, "| Test:", X_test.shape)


Train: (1069, 9) | Test: (268, 9)


## 2. Model 1 - Linear Regression (Baseline)
Trained on scaled features. We tested a log-transform of the target and it made
results worse (see Notebook 1, Section 12), so we train directly on `charges`.

In [2]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

print("Linear Regression")
print("MAE :", mean_absolute_error(y_test, y_pred_lr))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr)))
print("R2  :", r2_score(y_test, y_pred_lr))


Linear Regression
MAE : 2828.973747103939
RMSE: 4572.81123935467
R2  : 0.8862045574393046


## 3. Model 2 - Decision Tree Regressor
Hyperparameter tuning via GridSearchCV (expanded grid).

In [3]:
param_grid_dt = {
    "max_depth": [3, 5, 7, 10, 15, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

grid_dt = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid_dt,
    cv=5,
    scoring="r2",
    n_jobs=-1,
)
grid_dt.fit(X_train, y_train)

best_dt = grid_dt.best_estimator_
y_pred_dt = best_dt.predict(X_test)

print("Best Params:", grid_dt.best_params_)
print("MAE :", mean_absolute_error(y_test, y_pred_dt))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_dt)))
print("R2  :", r2_score(y_test, y_pred_dt))


Best Params: {'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 2}
MAE : 2693.266420693201
RMSE: 4461.371691390971
R2  : 0.8916833712530271


## 4. Model 3 - Random Forest Regressor
Hyperparameter tuning via GridSearchCV (expanded grid).

In [4]:
param_grid_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4],
}

grid_rf = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid_rf,
    cv=5,
    scoring="r2",
    n_jobs=-1,
)
grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)

print("Best Params:", grid_rf.best_params_)
print("MAE :", mean_absolute_error(y_test, y_pred_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))
print("R2  :", r2_score(y_test, y_pred_rf))


Best Params: {'max_depth': 5, 'min_samples_leaf': 4, 'n_estimators': 200}
MAE : 2414.52935034516
RMSE: 4261.848720693212
R2  : 0.9011550741856774


## 5. Documented Hyperparameters

In [5]:
print("Decision Tree best params:", grid_dt.best_params_)
print("Random Forest best params :", grid_rf.best_params_)


Decision Tree best params: {'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 2}
Random Forest best params : {'max_depth': 5, 'min_samples_leaf': 4, 'n_estimators': 200}


## 6. Save Trained Models

In [6]:
joblib.dump(lr, f"{MODELS_DIR}/linear_regression.pkl")
joblib.dump(best_dt, f"{MODELS_DIR}/decision_tree.pkl")
joblib.dump(best_rf, f"{MODELS_DIR}/random_forest.pkl")

print("All 3 models saved in ../models/")


All 3 models saved in ../models/


## Deliverables - Member 3
- [x] 3 trained models saved in `models/`
- [x] Hyperparameters documented (Section 5)
- [x] Expanded GridSearchCV for Decision Tree / Random Forest
- [x] Interaction feature (bmi_smoker) improved Linear Regression R2 from 0.81 to 0.89
